<h1> CNN-Based Classification Model Tutorial </h1>
<h5> <p style="font-size:90% ; font-family:arial"> (1) options.py </p> </h5> 
<h5> <p style="font-size:90% ; line-height:50%">  (2) pipeline.py </p> </h5> 
<h5> <p style="font-size:90% ; line-height:50%">  (3) networks.py </p> </h5> 
<h5> <p style="font-size:120% ; line-height:50% ; color:blue ; font-weight:bold">  (4) train.py </p> </h5> 

In [1]:
import os
import sys

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    sys.path.append("/content/drive/MyDrive/CBNU/Classification")
    print(os.getcwd())
    os.chdir("/content/drive/MyDrive/CBNU/Classification")
    print(os.getcwd())
except ModuleNotFoundError:
    print("Not in colab, continue")

Not in colab, continue


위 내용은 앞에서 다뤘습니다. <br>

In [2]:
import torch
import numpy as np

외부 라이브러리를 불러옵니다. <br>

In [3]:
from options import TrainOptions
opt = TrainOptions().parse()

TrainOptions를 불러와 선언합니다.

In [4]:
from networks import define_network, define_criterion, define_optimizer
from pipeline import define_dataset
from utils import fix_seed, get_num_params

networks.py 에서 define_network, define_criterion, define_optimizer를, <br> <br>
pipeline.py 에서 define_dataset을, <br> <br>
utils.py 에서 fix_seed, get_num_params를 불러옵니다. <br> <br>


In [5]:
fix_seed(opt.seed)

랜덤시드를 고정합니다. 시드 값은 option에 정의되어 있습니다.

In [6]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(device)

mps


device를 정의 합니다. <br> <br>
현재 실행 환경에 맞는 device가 출력됩니다. <br> <br>

In [8]:
network = define_network(opt).to(device)
criterion = define_criterion(opt).to(device)
optimizer = define_optimizer(network, opt)
print(network)
print(f"Number of parameters : {get_num_params(network)}")
print(criterion)
print(optimizer)

CustomNetwork(
  (classifier): Sequential(
    (0): Linear(in_features=25088, out_features=4096, bias=True)
    (1): ReLU(inplace=True)
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=4096, out_features=4096, bias=True)
    (4): ReLU(inplace=True)
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=4096, out_features=2, bias=True)
    (7): Softmax(dim=1)
  )
  (feature_extractor): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, 

network. criterion, optimizer를 선언합니다.

In [7]:
dataset, dataloader = define_dataset(opt)
print(len(dataset), len(dataloader))

3914 979


dataset, dataloader를 선언합니다.

In [9]:
save_dir = os.path.join(opt.save_root, opt.name)
os.makedirs(save_dir, exist_ok=True)

학습 결과를 저장할 경로를 선언하고 경로에 폴더를 생성합니다.